In [3]:
import json
import networkx as nx
import pandas as pd

# Load JSON files
with open(r"C:\Users\tuanl\collaboration-network-of-NTU-CCDS-faculty\main_authors.json", "r", encoding="utf-8") as f:
    main_authors = json.load(f)

with open(r"C:\Users\tuanl\collaboration-network-of-NTU-CCDS-faculty\all_collaborations.json", "r", encoding="utf-8") as f:
    all_collaborations = json.load(f)

# Set of NTU faculty PIDs
faculty_pids = {author["pid"] for author in main_authors}

# Step 1: Build undirected graph from all_collaborations.json
G = nx.Graph()

for author_pid, data in all_collaborations.items():
    author_name = data["author"]["name"]
    G.add_node(author_pid, name=author_name)

    for year, collaborators in data["collaborations_by_year"].items():
        for collab in collaborators:
            collab_pid = collab["pid"]
            collab_name = collab["name"]

            if collab_pid != author_pid:
                G.add_node(collab_pid, name=collab_name)
                G.add_edge(author_pid, collab_pid)

# Step 2: Identify external co-authors
external_coauthors = {n for n in G.nodes() if n not in faculty_pids}

# Step 3: Compute centrality measures
pagerank_scores = nx.pagerank(G, alpha=0.85)
degree_centrality_scores = nx.degree_centrality(G)

# Step 4: Combine centrality scores with normalization
combined_scores = []
for pid in external_coauthors:
    name = G.nodes[pid].get("name", pid)
    pagerank = pagerank_scores.get(pid, 0)
    degree = degree_centrality_scores.get(pid, 0)
    combined_scores.append({
        "pid": pid,
        "name": name,
        "pagerank": pagerank,
        "degree": degree
    })

df_combined = pd.DataFrame(combined_scores)

# Normalize pagerank and degree
df_combined['norm_pagerank'] = (df_combined['pagerank'] - df_combined['pagerank'].min()) / (df_combined['pagerank'].max() - df_combined['pagerank'].min())
df_combined['norm_degree'] = (df_combined['degree'] - df_combined['degree'].min()) / (df_combined['degree'].max() - df_combined['degree'].min())

# Calculate normalized combined score
df_combined['combined_score'] = 0.6 * df_combined['norm_pagerank'] + 0.4 * df_combined['norm_degree']

# Step 5: Generate three rankings
top_pagerank = df_combined.sort_values(by="pagerank", ascending=False).head(1000)
top_degree = df_combined.sort_values(by="degree", ascending=False).head(1000)
top_combined = df_combined.sort_values(by="combined_score", ascending=False).head(1000)

# Save to CSV or use for downstream analysis
top_pagerank.to_csv("top_1000_pagerank.csv", index=False)
top_degree.to_csv("top_1000_degree.csv", index=False)
top_combined.to_csv("top_1000_combined.csv", index=False)

# Step 6: Identify entries in pagerank/degree lists but not in combined
pagerank_only = top_pagerank[~top_pagerank["pid"].isin(top_combined["pid"])].copy()
pagerank_only["source"] = "pagerank_only"

degree_only = top_degree[~top_degree["pid"].isin(top_combined["pid"])].copy()
degree_only["source"] = "degree_only"

unique_contributors = pd.concat([pagerank_only, degree_only], ignore_index=True)
unique_contributors.to_csv("unique_not_in_combined.csv", index=False)

print("Top 1000 candidates exported by PageRank, Degree Centrality, and Combined Score.")
print("Also saved unique contributors not in combined list.")


Top 1000 candidates exported by PageRank, Degree Centrality, and Combined Score.
Also saved unique contributors not in combined list.
